
### Workaround - eigen muziek

In de spotify file (standaard - automatische generatie download). Kan je de naam van playlists vinden alsook hun nummers en respectievelijke artisten. Dit wordt binnen de 5 dagen beschikbaar gesteld voor dowload en is een CSV file, zoals mijn data reeds aanwezig. Dit kan gebruikt worden om een database op te zetten met iedereen zijn nummers. En op deze nummer database (uitgebreid met de features van de nummers) kan dan aan playlistgeneratie gedaan worden - To be continued

TODO: Met behulp van copiloot een programma schrijven die de nummers uit bepaalde playlists haalt en in een database structuur bewaart. Waarbij de persoon vermeld wordt als 'bezitter' van de muziek in een playlist. One - to -many relatie. Een song kan het 'bezit' zijn van meerdere mensen

✅ TODO: Playlists aan bibliotheek toevoegen van de deelnemers en info opvragen aan spotify om te zien of ik zo ook aan hun playlist nummers kan

### Scraping zelfgemaakte playlists

In [1]:
import os
import pandas as pd
import json
from pathlib import Path
import json


In [2]:
# 1. Basismap instellen (de map waar Spotify_data staat)
base_dir = Path(r"C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data\Brons\Spotify_data")

print("Base dir:", base_dir)
print("Bestaat base_dir?:", base_dir.exists())

# 2. Submap account_log definiëren
account_dir = base_dir / "account_log"

print("Account dir:", account_dir)
print("Bestaat account_dir?:", account_dir.exists())


Base dir: C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data\Brons\Spotify_data
Bestaat base_dir?: True
Account dir: C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data\Brons\Spotify_data\account_log
Bestaat account_dir?: True


In [3]:
# Controleren of de account_log map bestaat
if not account_dir.exists():
    raise FileNotFoundError(f"Map bestaat niet: {account_dir}")

# Alle bestanden in account_log oplijsten
files_in_account = list(account_dir.iterdir())

print(f"Aantal bestanden in account_log: {len(files_in_account)}")
for f in files_in_account:
    print("-", f.name)


Aantal bestanden in account_log: 16
- Follow.json
- Identifiers.json
- Identity.json
- Marquee.json
- Payments.json
- Playlist1.json
- PodcastInteractivityRatedShow.json
- Read_Me_First.pdf
- SearchQueries.json
- StreamingHistory_audiobook_0.json
- StreamingHistory_music_0.json
- StreamingHistory_podcast_0.json
- Userdata.json
- Wrapped2024.json
- YourLibrary.json
- YourSoundCapsule.json


In [4]:
# 1. Pad naar het specifieke JSON-bestand
playlist_file = account_dir / "Playlist1.json"

print("Bestand:", playlist_file)
print("Bestaat bestand?:", playlist_file.exists())

# 2. Veilig openen met try/except
data = None
try:
    with open(playlist_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("JSON succesvol ingelezen.")
except json.JSONDecodeError as e:
    print("JSON decode error:", e)
except Exception as e:
    print("Andere fout:", e)

# 3. Eerste laag tonen (structuur, geen volledige inhoud)
if isinstance(data, dict):
    print("Top-level keys:", list(data.keys()))
elif isinstance(data, list):
    print("Top-level is een lijst met lengte:", len(data))
    if len(data) > 0 and isinstance(data[0], dict):
        print("Keys van eerste element:", list(data[0].keys()))
else:
    print("Onbekende datastructuur:", type(data))


Bestand: C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data\Brons\Spotify_data\account_log\Playlist1.json
Bestaat bestand?: True
JSON succesvol ingelezen.
Top-level keys: ['playlists']


In [5]:
# Stap 4: structuur van playlists inspecteren

# 1. Controleren dat 'playlists' bestaat
if "playlists" not in data:
    raise KeyError("Key 'playlists' niet gevonden in Playlist1.json")

playlists = data["playlists"]

# 2. Type en lengte tonen
print("Type van playlists:", type(playlists))
print("Aantal playlists:", len(playlists))

# 3. Eerste element inspecteren (alleen keys)
if len(playlists) > 0:
    first = playlists[0]
    print("Type van eerste playlist:", type(first))
    if isinstance(first, dict):
        print("Keys van eerste playlist:", list(first.keys()))
    else:
        print("Eerste playlist is geen dict:", type(first))
else:
    print("Geen playlists gevonden.")


Type van playlists: <class 'list'>
Aantal playlists: 3
Type van eerste playlist: <class 'dict'>
Keys van eerste playlist: ['name', 'lastModifiedDate', 'collaborators', 'items', 'description', 'numberOfFollowers']


In [6]:
# Stap 5: structuur van items in de eerste playlist inspecteren

first_playlist = playlists[0]

# 1. Controleren dat 'items' bestaat
if "items" not in first_playlist:
    raise KeyError("Key 'items' niet gevonden in de eerste playlist")

items = first_playlist["items"]

# 2. Type en lengte tonen
print("Type van items:", type(items))
print("Aantal items:", len(items))

# 3. Eerste item inspecteren
if len(items) > 0:
    first_item = items[0]
    print("Type van eerste item:", type(first_item))
    if isinstance(first_item, dict):
        print("Keys van eerste item:", list(first_item.keys()))
    else:
        print("Eerste item is geen dict:", type(first_item))
else:
    print("Geen items gevonden in deze playlist.")


Type van items: <class 'list'>
Aantal items: 110
Type van eerste item: <class 'dict'>
Keys van eerste item: ['track', 'episode', 'audiobook', 'localTrack', 'addedDate']


In [7]:
# Stap 6: structuur van de track in het eerste item inspecteren

first_item = items[0]

# 1. Controleren of er een track aanwezig is
track = first_item.get("track")

if track is None:
    print("Dit item bevat geen track (misschien een episode of audiobook).")
else:
    print("Type van track:", type(track))
    print("Keys van track:", list(track.keys()))


Type van track: <class 'dict'>
Keys van track: ['trackName', 'artistName', 'albumName', 'trackUri']


In [8]:
# Stap 7: controleren of alle items in alle playlists een track bevatten

total_items = 0
items_with_track = 0
items_without_track = 0

for pl in playlists:
    for item in pl.get("items", []):
        total_items += 1
        if item.get("track") is not None:
            items_with_track += 1
        else:
            items_without_track += 1

print("Totaal aantal items:", total_items)
print("Items met track:", items_with_track)
print("Items zonder track:", items_without_track)


Totaal aantal items: 157
Items met track: 157
Items zonder track: 0


In [9]:
# Stap 8: waarden van de eerste track inspecteren

first_track = items[0].get("track")

print("Track name:", first_track.get("trackName"))
print("Artist name:", first_track.get("artistName"))
print("Album name:", first_track.get("albumName"))
print("Track URI:", first_track.get("trackUri"))


Track name: Tomorrow
Artist name: Avril Lavigne
Album name: Let Go
Track URI: spotify:track:1T0Dv84b0Za7FoLcoeajgh


In [10]:
# Stap 9: unieke track-ID extraheren uit de URI

uri = first_track.get("trackUri")

if uri is None:
    raise ValueError("Geen trackUri gevonden in deze track.")

parts = uri.split(":")

if len(parts) < 3:
    raise ValueError(f"Onverwachte URI-structuur: {uri}")

track_id = parts[-1]

print("Volledige URI:", uri)
print("Geëxtraheerde track-ID:", track_id)


Volledige URI: spotify:track:1T0Dv84b0Za7FoLcoeajgh
Geëxtraheerde track-ID: 1T0Dv84b0Za7FoLcoeajgh


In [11]:
# Stap 10: controleren of alle tracks een geldige URI hebben

total_tracks = 0
valid_uri = 0
invalid_uri = 0
invalid_examples = []

for pl in playlists:
    for item in pl.get("items", []):
        track = item.get("track")
        if track is None:
            continue  # zou niet mogen gebeuren volgens vorige stap

        total_tracks += 1
        uri = track.get("trackUri")

        # Check 1: bestaat de URI?
        if uri is None:
            invalid_uri += 1
            invalid_examples.append(("missing", track))
            continue

        # Check 2: correcte structuur?
        parts = uri.split(":")
        if len(parts) == 3 and parts[0] == "spotify" and parts[1] == "track":
            valid_uri += 1
        else:
            invalid_uri += 1
            invalid_examples.append((uri, track))

print("Totaal aantal tracks:", total_tracks)
print("Tracks met geldige URI:", valid_uri)
print("Tracks met ongeldige of ontbrekende URI:", invalid_uri)

# Toon max 3 voorbeelden van problemen
if invalid_examples:
    print("\nVoorbeelden van ongeldige URI's:")
    for ex in invalid_examples[:3]:
        print("-", ex[0])


Totaal aantal tracks: 157
Tracks met geldige URI: 157
Tracks met ongeldige of ontbrekende URI: 0


In [12]:
item = items[0]
track = item["track"]

# Unieke ID extraheren
uri = track["trackUri"]
track_id = uri.split(":")[-1]

row = {
    "track_id": track_id,
    "track_name": track["trackName"],
    "artist_name": track["artistName"],
    "owner": "Astrid"  # voorlopig hardcoded, later dynamisch
}

row


{'track_id': '1T0Dv84b0Za7FoLcoeajgh',
 'track_name': 'Tomorrow',
 'artist_name': 'Avril Lavigne',
 'owner': 'Astrid'}

In [13]:
# Stap 12: alle tracks uit alle playlists extraheren naar een lijst van dicts

all_tracks = []

for pl in playlists:
    for item in pl.get("items", []):
        track = item.get("track")
        if track is None:
            continue  # zou niet mogen gebeuren, maar we blijven veilig

        # Unieke ID extraheren
        uri = track.get("trackUri")
        if uri is None:
            continue  # overslaan als er geen URI is

        parts = uri.split(":")
        if len(parts) < 3:
            continue  # ongeldige structuur, overslaan

        track_id = parts[-1]

        # Rij bouwen
        row = {
            "track_id": track_id,
            "track_name": track.get("trackName"),
            "artist_name": track.get("artistName"),
            "owner": "Astrid"
        }

        all_tracks.append(row)

# Resultaat tonen
print("Aantal geëxtraheerde tracks:", len(all_tracks))
print("Eerste track:", all_tracks[0])


Aantal geëxtraheerde tracks: 157
Eerste track: {'track_id': '1T0Dv84b0Za7FoLcoeajgh', 'track_name': 'Tomorrow', 'artist_name': 'Avril Lavigne', 'owner': 'Astrid'}


In [14]:
# Stap 13: lijst van tracks omzetten naar DataFrame
df_tracks = pd.DataFrame(all_tracks)

print("Vorm van DataFrame:", df_tracks.shape)
df_tracks.head()


Vorm van DataFrame: (157, 4)


,track_id,track_name,artist_name,owner
0,1T0Dv84b0Za7FoLcoeajgh,Tomorrow,Avril Lavigne,Astrid
1,2tcCAty2pDlQWeb8TEeS6R,Virgin State Of Mind,K's Choice,Astrid
2,0Bo7grcJ9Api986n2RNcA8,Call It Off,Tegan and Sara,Astrid
3,2LLcI3oU4CedCQDV5tJ1SK,Concrete Angel,Martina McBride,Astrid
4,3P4skIO0EF1c3C93GNwQpa,My Number,Tegan and Sara,Astrid


In [15]:
# Stap 14: controleren op duplicaten

# 1. Exacte duplicaten (volledige rijen)
exact_dupes = df_tracks.duplicated().sum()
print("Aantal exacte duplicaten:", exact_dupes)

# 2. Duplicaten op track_id
id_dupes = df_tracks.duplicated(subset=["track_id"]).sum()
print("Aantal duplicaten op track_id:", id_dupes)

# 3. Indien duplicaten bestaan, toon enkele voorbeelden
if id_dupes > 0:
    print("\nVoorbeelden van duplicaten op track_id:")
    display(df_tracks[df_tracks.duplicated(subset=["track_id"], keep=False)].head())


Aantal exacte duplicaten: 2
Aantal duplicaten op track_id: 2

Voorbeelden van duplicaten op track_id:


,track_id,track_name,artist_name,owner
85,0TDLuuLlV54CkRRUOahJb4,Titanium (feat. Sia),David Guetta,Astrid
116,0TDLuuLlV54CkRRUOahJb4,Titanium (feat. Sia),David Guetta,Astrid
118,07GvNcU1WdyZJq3XxP0kZa,Go Your Own Way - 2004 Remaster,Fleetwood Mac,Astrid
128,07GvNcU1WdyZJq3XxP0kZa,Go Your Own Way - 2004 Remaster,Fleetwood Mac,Astrid


In [16]:
# Stap 15: duplicaten verwijderen op basis van track_id

df_unique = df_tracks.drop_duplicates(subset=["track_id"], keep="first")

print("Vorm originele DataFrame:", df_tracks.shape)
print("Vorm zonder duplicaten:", df_unique.shape)

df_unique.head()


Vorm originele DataFrame: (157, 4)
Vorm zonder duplicaten: (155, 4)


,track_id,track_name,artist_name,owner
0,1T0Dv84b0Za7FoLcoeajgh,Tomorrow,Avril Lavigne,Astrid
1,2tcCAty2pDlQWeb8TEeS6R,Virgin State Of Mind,K's Choice,Astrid
2,0Bo7grcJ9Api986n2RNcA8,Call It Off,Tegan and Sara,Astrid
3,2LLcI3oU4CedCQDV5tJ1SK,Concrete Angel,Martina McBride,Astrid
4,3P4skIO0EF1c3C93GNwQpa,My Number,Tegan and Sara,Astrid


In [17]:
# Stap 16: controleren hoeveel owners er zijn en of tracks meerdere owners hebben

unique_owners = df_unique["owner"].unique()
print("Unieke owners:", unique_owners)

# Controleren of één track meerdere owners heeft
owner_counts = df_unique.groupby("track_id")["owner"].nunique()
multi_owner_tracks = owner_counts[owner_counts > 1]

print("Aantal tracks met meerdere owners:", len(multi_owner_tracks))

if len(multi_owner_tracks) > 0:
    print("\nVoorbeelden van tracks met meerdere owners:")
    display(df_unique[df_unique["track_id"].isin(multi_owner_tracks.index)].head())


Unieke owners: ['Astrid']
Aantal tracks met meerdere owners: 0


In [32]:
print("base_dir:", base_dir)
print("base_dir.name:", base_dir.name)
print("base_dir.parent:", base_dir.parent)




base_dir: C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data\Brons\Spotify_data
base_dir.name: Spotify_data
base_dir.parent: C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data\Brons


In [33]:
# base_dir = .../Data/Brons/Spotify_data
data_dir = base_dir.parent.parent   # .../Data
print("Gevonden data_dir:", data_dir)

# Zilver-map onder Data
zilver_dir = data_dir / "Zilver"
zilver_dir.mkdir(exist_ok=True)

# Submap voor Spotify-data binnen Zilver
zilver_spotify_dir = zilver_dir / "spotify_data"
zilver_spotify_dir.mkdir(exist_ok=True)

# Bestandsnaam voor de playlists
output_file_playlist = zilver_spotify_dir / "playlist_astrid.xlsx"

# Wegschrijven zonder index
df_unique.to_excel(output_file_playlist, index=False)

print("Playlist-bestand weggeschreven naar:")
print(output_file_playlist)
print("Bestaat output-bestand?:", output_file_playlist.exists())


Gevonden data_dir: C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data
Playlist-bestand weggeschreven naar:
C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data\Zilver\spotify_data\playlist_astrid.xlsx
Bestaat output-bestand?: True


### Volledige extractie van gelikete songs

In [19]:
library_file = account_dir / "YourLibrary.json"

print("Bestand:", library_file)
print("Bestaat bestand?:", library_file.exists())

data_lib = None
try:
    with open(library_file, "r", encoding="utf-8") as f:
        data_lib = json.load(f)
    print("JSON succesvol ingelezen.")
except json.JSONDecodeError as e:
    print("JSON decode error:", e)
except Exception as e:
    print("Andere fout:", e)

# Top-level structuur tonen
if isinstance(data_lib, dict):
    print("Top-level keys:", list(data_lib.keys()))
elif isinstance(data_lib, list):
    print("Top-level is een lijst met lengte:", len(data_lib))
else:
    print("Onbekende datastructuur:", type(data_lib))


Bestand: C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data\Brons\Spotify_data\account_log\YourLibrary.json
Bestaat bestand?: True
JSON succesvol ingelezen.
Top-level keys: ['tracks', 'albums', 'shows', 'episodes', 'bannedTracks', 'artists', 'bannedArtists', 'other']


In [20]:
# Stap 2: inspecteren van het aantal gelikete tracks

library_tracks = data_lib.get("tracks", [])

print("Aantal gelikete tracks:", len(library_tracks))

# Inspecteer de structuur van de eerste track
if len(library_tracks) > 0:
    print("Type van eerste track:", type(library_tracks[0]))
    print("Keys van eerste track:", list(library_tracks[0].keys()))
else:
    print("Geen tracks gevonden in YourLibrary.json")


Aantal gelikete tracks: 677
Type van eerste track: <class 'dict'>
Keys van eerste track: ['artist', 'album', 'track', 'uri']


In [21]:
# Stap 3: inspecteren van de inhoud van één gelikete track

first_lib_track = library_tracks[0]

print("Artist:", first_lib_track.get("artist"))
print("Album:", first_lib_track.get("album"))
print("Track:", first_lib_track.get("track"))
print("URI:", first_lib_track.get("uri"))


Artist: 2Pac
Album: Loyal To The Game
Track: Ghetto Gospel
URI: spotify:track:7jLbTp3qZzah9kMIdW8e5M


In [22]:
# Stap 4: track-ID extraheren uit de URI van de eerste library-track

uri = first_lib_track.get("uri")

parts = uri.split(":")
track_id = parts[-1] if len(parts) >= 3 else None

print("Volledige URI:", uri)
print("Geëxtraheerde track-ID:", track_id)


Volledige URI: spotify:track:7jLbTp3qZzah9kMIdW8e5M
Geëxtraheerde track-ID: 7jLbTp3qZzah9kMIdW8e5M


In [23]:
# Stap 5: alle gelikete tracks extraheren naar een lijst van dicts

library_rows = []

for item in library_tracks:
    uri = item.get("uri")
    if not uri:
        continue

    parts = uri.split(":")
    if len(parts) < 3:
        continue

    track_id = parts[-1]

    row = {
        "track_id": track_id,
        "track_name": item.get("track"),
        "artist_name": item.get("artist"),
        "owner": "Astrid"
    }

    library_rows.append(row)

print("Aantal geëxtraheerde library-tracks:", len(library_rows))
print("Eerste rij:", library_rows[0])


Aantal geëxtraheerde library-tracks: 677
Eerste rij: {'track_id': '7jLbTp3qZzah9kMIdW8e5M', 'track_name': 'Ghetto Gospel', 'artist_name': '2Pac', 'owner': 'Astrid'}


In [24]:
# Stap 6: DataFrame maken
df_library = pd.DataFrame(library_rows)

print("Vorm originele library-DataFrame:", df_library.shape)
print(df_library.head())

# Duplicaten verwijderen
df_library_unique = df_library.drop_duplicates(subset=["track_id"], keep="first")

print("\nVorm zonder duplicaten:", df_library_unique.shape)
df_library_unique.head()


Vorm originele library-DataFrame: (677, 4)
                 track_id             track_name     artist_name   owner
0  7jLbTp3qZzah9kMIdW8e5M          Ghetto Gospel            2Pac  Astrid
1  27ilMN1oiW9849fReEYgOj              I Love It       Icona Pop  Astrid
2  3MVvjJMdyzMzlMGbBhdkO4             Vale Decem     Murray Gold  Astrid
3  6qUcJCdjOi2OtSMBSvR3Vb                 Notion    Tash Sultana  Astrid
4  3VtlSKbs0IjVeeRV4otHNT  Love Songs For Robots  Patrick Watson  Astrid

Vorm zonder duplicaten: (677, 4)


,track_id,track_name,artist_name,owner
0,7jLbTp3qZzah9kMIdW8e5M,Ghetto Gospel,2Pac,Astrid
1,27ilMN1oiW9849fReEYgOj,I Love It,Icona Pop,Astrid
2,3MVvjJMdyzMzlMGbBhdkO4,Vale Decem,Murray Gold,Astrid
3,6qUcJCdjOi2OtSMBSvR3Vb,Notion,Tash Sultana,Astrid
4,3VtlSKbs0IjVeeRV4otHNT,Love Songs For Robots,Patrick Watson,Astrid


In [34]:
# Zilver-map aanmaken naast Brons
# base_dir = .../Data/Brons/Spotify_data
data_dir = base_dir.parent.parent   # .../Data
print("Gevonden data_dir:", data_dir)

# Zilver-map onder Data
zilver_dir = data_dir / "Zilver"
zilver_dir.mkdir(exist_ok=True)

# Submap voor Spotify-data binnen Zilver
zilver_spotify_dir = zilver_dir / "spotify_data"
zilver_spotify_dir.mkdir(exist_ok=True)

# Bestandsnaam voor de library
output_file_library = zilver_spotify_dir / "library_astrid.xlsx"

# Wegschrijven zonder index
df_library_unique.to_excel(output_file_library, index=False)

print("Library-bestand weggeschreven naar:")
print(output_file_library)
print("Bestaat output-bestand?:", output_file_library.exists())



Gevonden data_dir: C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data
Library-bestand weggeschreven naar:
C:\Users\astri\Desktop\Data_Scientist\Eindwerk\Data\Zilver\spotify_data\library_astrid.xlsx
Bestaat output-bestand?: True


### Here be dragons

In [26]:
import selenium
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

In [27]:
CHROMEDRIVER_PATH = r"C:\tools\chromedriver\chromedriver.exe"


In [28]:
os.path.exists(CHROMEDRIVER_PATH)


True

In [29]:
service = Service(CHROMEDRIVER_PATH)
driver = webdriver.Chrome(service=service)

driver.get("https://www.google.com")


In [30]:
playlist_url = input("Plak hier je Spotify playlist URL: ").strip()
playlist_url


''

In [31]:
driver.get(playlist_url)


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=143.0.7499.170); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff64e6488e5
	0x7ff64e648940
	0x7ff64e42165d
	0x7ff64e40d202
	0x7ff64e4327af
	0x7ff64e4a9a29
	0x7ff64e4ca5c2
	0x7ff64e46ac29
	0x7ff64e46ba93
	0x7ff64e960640
	0x7ff64e95af80
	0x7ff64e9796e6
	0x7ff64e665de4
	0x7ff64e66ed8c
	0x7ff64e652004
	0x7ff64e6521b5
	0x7ff64e637ee2
	0x7ff95beae8d7
	0x7ff95ce8c53c
